# UFC Data Cleaning

| Cell | What | Output |
|------|------|--------|
| 1 | Imports & constants | — |
| 2 | Load raw data | — |
| 3 | Year filter + drop Draw/NC | — |
| 4 | Clean events | `data/events_clean.csv` |
| 5 | Clean fighters (directory + profiles) | `data/fighters_clean.csv` |
| 6 | Clean fights (JSON stats) | — |
| 7 | Save + verify | `data/fights_clean.csv` |

**F1 = Red corner = Favorite | F2 = Blue corner = Underdog**
**Baseline: ~57% red corner win rate**

In [1]:
# notebooks/02_data_cleaning.ipynb — Cell 1: Imports & Constants

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import re
import os

# Auto-detect data path
if os.path.exists('./data/fights.csv'):
    DATA_DIR = './data'
elif os.path.exists('../data/fights.csv'):
    DATA_DIR = '../data'
else:
    raise FileNotFoundError('Cannot find data/ directory')

CUTOFF_YEAR = 2015
print(f'DATA_DIR: {DATA_DIR}')
print(f'Cutoff year: {CUTOFF_YEAR}')

DATA_DIR: ./data
Cutoff year: 2015


In [2]:
# notebooks/02_data_cleaning.ipynb — Cell 2: Load Raw Data

events = pd.read_csv(f'{DATA_DIR}/events.csv')
fights = pd.read_csv(f'{DATA_DIR}/fights.csv')
fighters = pd.read_csv(f'{DATA_DIR}/fighters_full.csv')

with open(f'{DATA_DIR}/fight_details.json', 'r') as f:
    fight_details = json.load(f)

print(f'Events:        {events.shape}')
print(f'Fights:        {fights.shape}')
print(f'Fighters:      {fighters.shape}')
print(f'Fight details: {len(fight_details)}')

# Quick sanity check
decided = (fights['winner']==fights['fighter_1']) | (fights['winner']==fights['fighter_2'])
f1w = (fights['winner']==fights['fighter_1']).sum()
f2w = (fights['winner']==fights['fighter_2']).sum()
dnc = (~decided).sum()
print(f'\nAll-time: Red {f1w} | Blue {f2w} | Draw/NC {dnc}')
print(f'Red WR (decided): {f1w/(f1w+f2w):.1%}')

# Verify profile columns exist
for col in ['slpm','sapm','str_acc_pct','str_def_pct',
            'td_avg','td_acc_pct','td_def_pct','sub_avg','dob']:
    status = '✅' if col in fighters.columns else '❌ MISSING'
    if col in fighters.columns:
        sample = fighters[col].dropna().head(3).tolist()
        print(f'  {col:15s} {status}  sample: {sample}')
    else:
        print(f'  {col:15s} {status}')

Events:        (769, 4)
Fights:        (8637, 18)
Fighters:      (4486, 20)
Fight details: 8637

All-time: Red 5454 | Blue 3029 | Draw/NC 154
Red WR (decided): 64.3%
  slpm            ✅  sample: ['|0.00', '|3.29', '|3.00']
  sapm            ✅  sample: ['|0.00', '|4.41', '|5.67']
  str_acc_pct     ✅  sample: ['|0%', '|38%', '|20%']
  str_def_pct     ✅  sample: ['|0%', '|57%', '|46%']
  td_avg          ✅  sample: ['|0.00', '|0.00', '|0.00']
  td_acc_pct      ✅  sample: ['|0%', '|0%', '|0%']
  td_def_pct      ✅  sample: ['|0%', '|77%', '|66%']
  sub_avg         ✅  sample: ['|0.0', '|0.0', '|0.0']
  dob             ✅  sample: ['|Jul 13, 1978', '|Jul 03, 1983', '|Feb 01, 1994']


In [3]:
# notebooks/02_data_cleaning.ipynb — Cell 3: Year Filter + Drop Draw/NC

# Parse event dates and map to fights
events['event_date_parsed'] = pd.to_datetime(
    events['event_date'], format='mixed', errors='coerce')
date_map = events.set_index('event_name')['event_date_parsed']
fights['event_date_parsed'] = fights['event_name'].map(date_map)
fights['year'] = fights['event_date_parsed'].dt.year

# Show red WR by era to justify cutoff
print('RED CORNER WIN RATE BY ERA:')
decided = fights[
    (fights['winner']==fights['fighter_1']) |
    (fights['winner']==fights['fighter_2'])
].copy()
decided['f1_win'] = (decided['winner']==decided['fighter_1']).astype(int)
for label, lo, hi in [('Pre-2010',0,2010),('2010-2014',2010,2015),
                      ('2015-2019',2015,2020),('2020+',2020,2100)]:
    s = decided[(decided['year']>=lo)&(decided['year']<hi)]
    if len(s)>0:
        print(f'  {label:12s} {len(s):>5d} fights  RedWR={s["f1_win"].mean():.1%}')

# Apply year filter
pre = len(fights)
fights = fights[fights['year'] >= CUTOFF_YEAR].reset_index(drop=True)
print(f'\nYear filter: {pre} -> {len(fights)}')

# Drop Draw/NC — direct comparison, NOT regex
# str.contains('NC') falsely matches Francisco, Duncan, Vince
decided_mask = (fights['winner']==fights['fighter_1']) | \
               (fights['winner']==fights['fighter_2'])
dnc = (~decided_mask).sum()
if dnc > 0:
    print(f'\nDropping {dnc} Draw/NC:')
    print(fights[~decided_mask][['fighter_1','fighter_2','winner']].to_string())
fights = fights[decided_mask].reset_index(drop=True)
print(f'After cleanup: {len(fights)} decided fights')
print(f'Red WR: {(fights["winner"]==fights["fighter_1"]).mean():.1%}')

RED CORNER WIN RATE BY ERA:
  Pre-2010      1250 fights  RedWR=100.0%
  2010-2014     1748 fights  RedWR=61.3%
  2015-2019     2368 fights  RedWR=56.7%
  2020+         3117 fights  RedWR=57.4%

Year filter: 8637 -> 5589

Dropping 104 Draw/NC:
                  fighter_1                  fighter_2   winner
9             Chris Padilla            MarQuel Mederos  Draw/NC
23              Ricky Simon               Adrian Yanez  Draw/NC
131          Jan Blachowicz              Bogdan Guskov  Draw/NC
145       Kennedy Nzechukwu            Marcus Buchecha  Draw/NC
174            Ode Osbourne               Alibi Idiris  Draw/NC
207            Tom Aspinall                 Ciryl Gane  Draw/NC
309              Zach Reese            Sedriques Dumas  Draw/NC
414      Mansur Abdul-Malik              Cody Brundage  Draw/NC
416              Paul Craig            Rodolfo Bellato  Draw/NC
604             Jimmy Crute            Rodolfo Bellato  Draw/NC
899    Abdul Razak Alhassan              Cody Brundag

In [4]:
# notebooks/02_data_cleaning.ipynb — Cell 4: Clean Events
# Output: data/events_clean.csv

events_clean = events.copy()
events_clean['event_date'] = pd.to_datetime(
    events_clean['event_date'], format='mixed', errors='coerce')
events_clean = events_clean.sort_values('event_date', ascending=False).reset_index(drop=True)
events_clean = events_clean[
    events_clean['event_date'].dt.year >= CUTOFF_YEAR
].reset_index(drop=True)

events_clean.to_csv(f'{DATA_DIR}/events_clean.csv', index=False)
print(f'Saved {len(events_clean)} events to data/events_clean.csv')
print(f'Range: {events_clean["event_date"].min()} to {events_clean["event_date"].max()}')

Saved 467 events to data/events_clean.csv
Range: 2015-01-03 00:00:00 to 2026-04-11 00:00:00


In [5]:
# notebooks/02_data_cleaning.ipynb — Cell 5: Clean Fighters
# Parses directory info (height/reach/weight) + profile stats (SLpM, etc.)
# Height/reach '--' → NaN, weight '-- lbs.' → NaN
# BUG FIX: scraper used get_text(separator='|') which left leading '|'
#   on all profile values: '|3.29', '|38%' etc.
#   All parse functions now strip '|' first via clean_val().
# Raw columns dropped after cleaning — only clean versions kept.
# Output: data/fighters_clean.csv

def clean_val(v):
    '''Strip leading | left by scraper get_text(separator="|")'''
    if pd.isna(v):
        return ''
    return str(v).replace('|','').strip()

def parse_height(h):
    h = clean_val(h)
    if not h or h == '--':
        return np.nan
    m = re.search(r"(\d+)'\s*(\d+)\"", h)
    return int(m.group(1))*12 + int(m.group(2)) if m else np.nan

def parse_reach(r):
    r = clean_val(r)
    if not r or r == '--':
        return np.nan
    m = re.search(r'([\d.]+)', r)
    return float(m.group(1)) if m else np.nan

def parse_weight(w):
    w = clean_val(w)
    if not w or w == '--':
        return np.nan
    m = re.search(r'([\d.]+)', w)
    return float(m.group(1)) if m else np.nan

def parse_pct(p):
    p = clean_val(p)
    if not p or p in ('--','0%'):
        return np.nan
    m = re.search(r'([\d.]+)', p)
    return float(m.group(1))/100.0 if m else np.nan

def parse_float(v):
    v = clean_val(v)
    if not v or v == '--':
        return np.nan
    try:
        return float(v)
    except ValueError:
        return np.nan

fc = fighters.copy()
fc['full_name'] = (fc['first_name'].fillna('')+' '+fc['last_name'].fillna('')).str.strip()

# Physical
fc['height_inches'] = fc['height'].apply(parse_height)
fc['reach_inches'] = fc['reach'].apply(parse_reach)
fc['weight_lbs'] = fc['weight'].apply(parse_weight)

# Record
for col in ['wins','losses','draws']:
    fc[col] = pd.to_numeric(fc[col], errors='coerce').fillna(0).astype(int)
fc['total_fights'] = fc['wins'] + fc['losses'] + fc['draws']
fc['win_pct'] = np.where(fc['total_fights']>0, fc['wins']/fc['total_fights'], 0)

# Profile career rate stats — strip | first
fc['slpm'] = fc['slpm'].apply(parse_float)
fc['sapm'] = fc['sapm'].apply(parse_float)
fc['str_acc_career'] = fc['str_acc_pct'].apply(parse_pct)
fc['str_def_career'] = fc['str_def_pct'].apply(parse_pct)
fc['td_avg'] = fc['td_avg'].apply(parse_float)
fc['td_acc_career'] = fc['td_acc_pct'].apply(parse_pct)
fc['td_def_career'] = fc['td_def_pct'].apply(parse_pct)
fc['sub_avg'] = fc['sub_avg'].apply(parse_float)

# DOB
fc['dob_parsed'] = pd.to_datetime(
    fc['dob'].apply(clean_val), format='mixed', errors='coerce')

# Drop raw duplicate columns — clean versions are kept
raw_dupes = ['height', 'weight', 'reach', 'dob',
             'str_acc_pct', 'str_def_pct', 'td_acc_pct', 'td_def_pct']
fc = fc.drop(columns=raw_dupes)

fighters_clean = fc
fighters_clean.to_csv(f'{DATA_DIR}/fighters_clean.csv', index=False)

print(f'Saved {len(fighters_clean)} fighters to data/fighters_clean.csv')
print(f'Columns: {list(fighters_clean.columns)}')
print(f'\nPhysical:')
for col in ['height_inches','reach_inches','weight_lbs','stance']:
    miss = fighters_clean[col].isnull().sum()
    print(f'  {col:15s} {miss} missing ({miss/len(fighters_clean)*100:.1f}%)')
print(f'\nProfile career stats:')
for col in ['slpm','sapm','str_acc_career','str_def_career',
            'td_avg','td_acc_career','td_def_career','sub_avg']:
    nn = fighters_clean[col].notna().sum()
    mn = fighters_clean[col].mean()
    print(f'  {col:18s} {nn:>5d} non-null  mean={mn:.3f}')
print(f'\nDOB: {fighters_clean["dob_parsed"].notna().sum()} parsed')

# Verify no raw pipe values remain
print(f'\nVerification — sample row:')
print(fighters_clean.iloc[1].to_string())

Saved 4486 fighters to data/fighters_clean.csv
Columns: ['fighter_url', 'first_name', 'last_name', 'nickname', 'stance', 'wins', 'losses', 'draws', 'slpm', 'sapm', 'td_avg', 'sub_avg', 'full_name', 'height_inches', 'reach_inches', 'weight_lbs', 'total_fights', 'win_pct', 'str_acc_career', 'str_def_career', 'td_acc_career', 'td_def_career', 'dob_parsed']

Physical:
  height_inches   347 missing (7.7%)
  reach_inches    1969 missing (43.9%)
  weight_lbs      86 missing (1.9%)
  stance          875 missing (19.5%)

Profile career stats:
  slpm                4486 non-null  mean=2.486
  sapm                4486 non-null  mean=3.231
  str_acc_career      3659 non-null  mean=0.439
  str_def_career      3727 non-null  mean=0.513
  td_avg              4486 non-null  mean=1.240
  td_acc_career       2607 non-null  mean=0.445
  td_def_career       2913 non-null  mean=0.616
  sub_avg             4486 non-null  mean=0.567

DOB: 3978 parsed

Verification — sample row:
fighter_url       http://www.u

In [6]:
# notebooks/02_data_cleaning.ipynb — Cell 6: Clean Fights
# Pull stats from fight_details.json (full X-of-Y format)
# JSON totals row: [0]=names [1]=KD [2]=sig_str [3]=sig_pct [4]=total_str
#                  [5]=TD [6]=TD_pct [7]=sub_att [8]=rev [9]=ctrl
# JSON sig strikes: [0]=names [1]=sig_str [2]=sig_pct [3]=head [4]=body
#                   [5]=leg [6]=distance [7]=clinch [8]=ground
# Raw method column dropped — only method_clean kept.

def parse_xofy(s):
    '''Parse "X of Y" → (landed, attempted)'''
    if pd.isna(s) or str(s).strip() in ('','--','---'):
        return np.nan, np.nan
    m = re.search(r'(\d+)\s+of\s+(\d+)', str(s))
    return (int(m.group(1)), int(m.group(2))) if m else (np.nan, np.nan)

def parse_ctrl(c):
    '''Parse "M:SS" → seconds'''
    if pd.isna(c) or str(c).strip() in ('','--','---'):
        return np.nan
    m = re.match(r'(\d+):(\d+)', str(c).strip())
    return int(m.group(1))*60 + int(m.group(2)) if m else np.nan

# Build lookup from JSON — totals + sig strike breakdown
detail_stats = {}
for fd in fight_details:
    url = fd.get('fight_url','')
    if not url:
        continue
    entry = {}

    # Totals table (Table 0)
    t_rows = fd.get('totals',{}).get('rows',[])
    if t_rows:
        r = t_rows[0]
        def sg(ci, fi):
            try: return r[ci][fi]
            except: return None
        entry.update({
            'f1_kd': sg(1,0), 'f2_kd': sg(1,1),
            'f1_sig_str': sg(2,0), 'f2_sig_str': sg(2,1),
            'f1_total_str': sg(4,0), 'f2_total_str': sg(4,1),
            'f1_td': sg(5,0), 'f2_td': sg(5,1),
            'f1_sub': sg(7,0), 'f2_sub': sg(7,1),
            'f1_rev': sg(8,0), 'f2_rev': sg(8,1),
            'f1_ctrl': sg(9,0), 'f2_ctrl': sg(9,1),
        })

    # Sig strikes breakdown (Table 1)
    s_rows = fd.get('significant_strikes',{}).get('rows',[])
    if s_rows:
        r2 = s_rows[0]
        def sg2(ci, fi):
            try: return r2[ci][fi]
            except: return None
        entry.update({
            'f1_head': sg2(3,0), 'f2_head': sg2(3,1),
            'f1_body': sg2(4,0), 'f2_body': sg2(4,1),
            'f1_leg': sg2(5,0), 'f2_leg': sg2(5,1),
            'f1_distance': sg2(6,0), 'f2_distance': sg2(6,1),
            'f1_clinch': sg2(7,0), 'f2_clinch': sg2(7,1),
            'f1_ground': sg2(8,0), 'f2_ground': sg2(8,1),
        })

    detail_stats[url] = entry

match = sum(1 for u in fights['fight_url'] if u in detail_stats)
print(f'JSON lookup: {len(detail_stats)} fights')
print(f'Match rate: {match}/{len(fights)} ({match/len(fights):.1%})')

# Build clean fights
fc = fights.copy()

# Map clean event dates
ed = events_clean.set_index('event_name')['event_date']
fc['event_date'] = fc['event_name'].map(ed)
fc['event_date'] = pd.to_datetime(fc['event_date'], format='mixed', errors='coerce')

# Pull all JSON stats
all_json_cols = [
    'f1_kd','f2_kd','f1_sig_str','f2_sig_str','f1_total_str','f2_total_str',
    'f1_td','f2_td','f1_sub','f2_sub','f1_rev','f2_rev','f1_ctrl','f2_ctrl',
    'f1_head','f2_head','f1_body','f2_body','f1_leg','f2_leg',
    'f1_distance','f2_distance','f1_clinch','f2_clinch','f1_ground','f2_ground',
]
for col in all_json_cols:
    fc[col] = fc['fight_url'].map(
        lambda u, c=col: detail_stats.get(u, {}).get(c))

# Parse sig strikes: X of Y → landed, attempted, accuracy
for p in ['f1','f2']:
    l, a = zip(*fc[f'{p}_sig_str'].apply(parse_xofy))
    fc[f'{p}_str_landed'] = pd.Series(l, dtype='float64')
    fc[f'{p}_str_attempted'] = pd.Series(a, dtype='float64')
    fc[f'{p}_str_acc'] = np.where(
        fc[f'{p}_str_attempted']>0,
        fc[f'{p}_str_landed']/fc[f'{p}_str_attempted'], 0)

# Parse total strikes
for p in ['f1','f2']:
    l, a = zip(*fc[f'{p}_total_str'].apply(parse_xofy))
    fc[f'{p}_total_str_landed'] = pd.Series(l, dtype='float64')
    fc[f'{p}_total_str_attempted'] = pd.Series(a, dtype='float64')

# Parse takedowns
for p in ['f1','f2']:
    l, a = zip(*fc[f'{p}_td'].apply(parse_xofy))
    fc[f'{p}_td_landed'] = pd.Series(l, dtype='float64')
    fc[f'{p}_td_attempted'] = pd.Series(a, dtype='float64')
    fc[f'{p}_td_acc'] = np.where(
        fc[f'{p}_td_attempted']>0,
        fc[f'{p}_td_landed']/fc[f'{p}_td_attempted'], 0)

# Parse sig strike location: head/body/leg/distance/clinch/ground
for p in ['f1','f2']:
    for loc in ['head','body','leg','distance','clinch','ground']:
        col = f'{p}_{loc}'
        l, a = zip(*fc[col].apply(parse_xofy))
        fc[f'{p}_{loc}_landed'] = pd.Series(l, dtype='float64')
        fc[f'{p}_{loc}_attempted'] = pd.Series(a, dtype='float64')

# Parse remaining numeric
for p in ['f1','f2']:
    fc[f'{p}_kd'] = pd.to_numeric(fc[f'{p}_kd'], errors='coerce')
    fc[f'{p}_sub'] = pd.to_numeric(fc[f'{p}_sub'], errors='coerce')
    fc[f'{p}_rev'] = pd.to_numeric(fc[f'{p}_rev'], errors='coerce')
    fc[f'{p}_ctrl_seconds'] = fc[f'{p}_ctrl'].apply(parse_ctrl)

# Round & time
fc['round'] = pd.to_numeric(fc['round'], errors='coerce')
def time_to_sec(t):
    if pd.isna(t) or str(t).strip() in ('','--'):
        return np.nan
    m = re.match(r'(\d+):(\d+)', str(t))
    return int(m.group(1))*60+int(m.group(2)) if m else np.nan
fc['time_seconds'] = fc['time'].apply(time_to_sec)
fc['total_time_seconds'] = ((fc['round']-1)*5*60) + fc['time_seconds']

# Target
fc['f1_win'] = (fc['winner']==fc['fighter_1']).astype(int)

# Clean method — strip embedded newlines from scraping
fc['method_clean'] = (fc['method']
    .str.replace(r'\s*\n+\s*', ' ', regex=True)
    .str.strip())
fc['finish_type'] = fc['method_clean'].str.upper().apply(lambda x:
    'KO/TKO' if 'KO' in str(x) else
    'SUB' if 'SUB' in str(x) else
    'DEC' if 'DEC' in str(x) else 'OTHER')
fc['weight_class'] = fc['weight_class'].str.strip()

# Drop raw/temp columns — clean versions are kept
drop = ['event_date_parsed','year','method',
        'f1_str','f2_str',
        'f1_sig_str','f2_sig_str','f1_total_str','f2_total_str',
        'f1_td','f2_td','f1_ctrl','f2_ctrl',
        'f1_head','f2_head','f1_body','f2_body','f1_leg','f2_leg',
        'f1_distance','f2_distance','f1_clinch','f2_clinch',
        'f1_ground','f2_ground']
fc = fc.drop(columns=[c for c in drop if c in fc.columns])

fights_clean = fc
print(f'Fights: {fights_clean.shape}')
print(f'Red WR: {fights_clean["f1_win"].mean():.1%}')
print(f'\nFinish types:')
print(fights_clean['finish_type'].value_counts().to_string())
print(f'\nColumns ({len(fights_clean.columns)}):')
print(list(fights_clean.columns))

JSON lookup: 8637 fights
Match rate: 5485/5485 (100.0%)
Fights: (5485, 62)
Red WR: 57.1%

Finish types:
finish_type
DEC       2737
KO/TKO    1760
SUB        975
OTHER       13

Columns (62):
['event_name', 'event_date', 'fight_url', 'fighter_1', 'fighter_2', 'winner', 'f1_kd', 'f2_kd', 'f1_sub', 'f2_sub', 'weight_class', 'round', 'time', 'f1_rev', 'f2_rev', 'f1_str_landed', 'f1_str_attempted', 'f1_str_acc', 'f2_str_landed', 'f2_str_attempted', 'f2_str_acc', 'f1_total_str_landed', 'f1_total_str_attempted', 'f2_total_str_landed', 'f2_total_str_attempted', 'f1_td_landed', 'f1_td_attempted', 'f1_td_acc', 'f2_td_landed', 'f2_td_attempted', 'f2_td_acc', 'f1_head_landed', 'f1_head_attempted', 'f1_body_landed', 'f1_body_attempted', 'f1_leg_landed', 'f1_leg_attempted', 'f1_distance_landed', 'f1_distance_attempted', 'f1_clinch_landed', 'f1_clinch_attempted', 'f1_ground_landed', 'f1_ground_attempted', 'f2_head_landed', 'f2_head_attempted', 'f2_body_landed', 'f2_body_attempted', 'f2_leg_landed', '

In [7]:
# notebooks/02_data_cleaning.ipynb — Cell 7: Save + Verify
# Output: data/fights_clean.csv

fights_clean.to_csv(f'{DATA_DIR}/fights_clean.csv', index=False)

print('=' * 60)
print('CLEANING COMPLETE')
print('=' * 60)
print(f'\nFiles saved to {DATA_DIR}/:')
print(f'  events_clean.csv    {len(events_clean):>5d} rows')
print(f'  fighters_clean.csv  {len(fighters_clean):>5d} rows')
print(f'  fights_clean.csv    {len(fights_clean):>5d} rows')

print(f'\nBaseline: Red WR = {fights_clean["f1_win"].mean():.1%}')

# Fight stat coverage
print(f'\nFight stat coverage:')
stat_groups = {
    'Striking': ['f1_str_landed','f1_str_attempted','f1_str_acc',
                 'f1_total_str_landed','f1_total_str_attempted'],
    'Grappling': ['f1_td_landed','f1_td_attempted','f1_td_acc',
                  'f1_kd','f1_sub','f1_rev','f1_ctrl_seconds'],
    'Location': ['f1_head_landed','f1_head_attempted',
                 'f1_body_landed','f1_body_attempted',
                 'f1_leg_landed','f1_leg_attempted'],
    'Position': ['f1_distance_landed','f1_distance_attempted',
                 'f1_clinch_landed','f1_clinch_attempted',
                 'f1_ground_landed','f1_ground_attempted'],
}
for group, cols in stat_groups.items():
    print(f'\n  {group}:')
    for col in cols:
        nn = fights_clean[col].notna().sum()
        mn = fights_clean[col].mean()
        print(f'    {col:30s} {nn:>5d}/{len(fights_clean)}  mean={mn:.2f}')

# Fighter career stat coverage
print(f'\nFighter career stat coverage:')
for col in ['slpm','sapm','str_acc_career','str_def_career',
            'td_avg','td_acc_career','td_def_career','sub_avg']:
    nn = fighters_clean[col].notna().sum()
    mn = fighters_clean[col].mean()
    print(f'  {col:18s} {nn:>5d}/{len(fighters_clean)}  mean={mn:.3f}')

print(f'\nDOB parsed: {fighters_clean["dob_parsed"].notna().sum()}/{len(fighters_clean)}')

# Verify no raw columns remain
print(f'\nFighters columns: {list(fighters_clean.columns)}')
print(f'Fights columns: {list(fights_clean.columns)}')

print(f'\nSample fights:')
print(fights_clean[['fighter_1','fighter_2','winner','f1_win',
    'f1_str_landed','f2_str_landed','f1_head_landed',
    'f1_body_landed','f1_leg_landed','finish_type']].head(10).to_string())

print(f'\nSample fighters:')
print(fighters_clean[['full_name','height_inches','reach_inches',
    'slpm','sapm','str_acc_career','str_def_career',
    'td_avg','td_acc_career','td_def_career','sub_avg']].head(5).to_string())

print(f'\nNext: open 03_eda.ipynb')

CLEANING COMPLETE

Files saved to ./data/:
  events_clean.csv      467 rows
  fighters_clean.csv   4486 rows
  fights_clean.csv     5485 rows

Baseline: Red WR = 57.1%

Fight stat coverage:

  Striking:
    f1_str_landed                   5485/5485  mean=16.31
    f1_str_attempted                5485/5485  mean=35.12
    f1_str_acc                      5485/5485  mean=0.48
    f1_total_str_landed             5485/5485  mean=22.44
    f1_total_str_attempted          5485/5485  mean=42.28

  Grappling:
    f1_td_landed                    5485/5485  mean=0.45
    f1_td_attempted                 5485/5485  mean=1.13
    f1_td_acc                       5485/5485  mean=0.22
    f1_kd                           5485/5485  mean=0.13
    f1_sub                          5485/5485  mean=0.14
    f1_rev                          5485/5485  mean=0.05
    f1_ctrl_seconds                 5485/5485  mean=54.08

  Location:
    f1_head_landed                  5485/5485  mean=9.57
    f1_head_attempted   